Implementar, a partir do código de BFS e DFS já visto em aula, o Aprofundamento Iterativo (IDS) com métricas de execução.

Analisar comparativamente BFS × DFS × IDS em instâncias do 8-puzzle.

Implemente o IDS (Aprofundamento Iterativo) no 8-puzzle
Use o mesmo código base que vocês já têm (BFS e DFS em tuplas 3×3, GOAL, valid_moves, etc.).
A tarefa é implementar do zero o Aprofundamento Iterativo (IDS), mantendo o mesmo estilo e métricas impressas dentro da função, como já fizemos com a BFS.

    Não copie um IDS pronto; use a lógica que derivamos de BFS/DFS.
    O retorno deve ser lista de estados (como na BFS), para você poder imprimir o caminho estado a estado.

Use estes três estados iniciais (todos solúveis e rápidos):

casos = {
    "Fácil":      (1,2,3,4,5,6,7,0,8),
    "Médio":      (1,2,3,4,5,6,0,7,8),
    "Desafiador": (1,2,3,5,0,6,4,7,8),
}


Monte uma tabela (no próprio notebook ou no PDF)

In [ ]:
from typing import List
import time

GOAL =  (
    1,2,3,
    4,5,6,
    7,8,0
)

casos = {
     "Fácil": (1,2,3,
               4,5,6,
               7,0,8),

     "Médio": (1,2,3,
               4,5,6,
               0,7,8),

     "Desafiador": (1,2,3,
                    5,0,6,
                    4,7,8)
}

# profundidade - custo para solução ser encontrada na arevore de busca
# nos expandidos - quantidade de nos expandidos - tirar da fronteira e gerar seus sucessores/adjacentes - quanto menos nos expandidos para achar a mesma solução, mais eficiente foi o algoritmo
# fronteira maxima - qual o tamanho maximo que a fronteira atinigiu durante o tempo de execução

class Resposta():
  def __init__(self, algoritmo: str, path: List[tuple], depth: int, expanded_nodes: int, max_frontier: int, time: float):
    self.algoritmo = algoritmo
    self.path = path
    self.depth = depth
    self.expanded_nodes = expanded_nodes
    self.max_frontier = max_frontier
    self.time = time


In [ ]:
MOVES = {
    "UP": -3,
    "DOWN": 3,
    "LEFT": -1,
    "RIGHT": 1,
}

def show(state: tuple)-> None:
  print(*(state[i:i+3] for i in range(0,9,3)))

def find_zero(state: tuple) -> int:
  return state.index(0)

def find_valid_moves(z: int) -> List[str]:
  moves = []
  r, c = divmod(z,3) # row and column

  # validar as regras de movimentação
  if r > 0:
      moves.append("UP")
  if r < 2:
      moves.append("DOWN")

  if c > 0:
      moves.append("LEFT")
  if c < 2:
      moves.append("RIGHT")

  return moves

# Função para aplicar a movimentação
def apply_moves(state: tuple, selected_movie: str, zero_index: int) -> tuple:
    new_zero_index = zero_index + MOVES[selected_movie]
    s = list(state) # permite movimentações de indice, transforma tupla em lista
    s[zero_index], s[new_zero_index] = s[new_zero_index], s[zero_index]
    return tuple(s)

In [ ]:
from collections import deque

def bfs(start: tuple) -> Resposta:
  frontiers = deque([[start]])
  visited = {start}

  # criando atributos de resposta
  start_time = time.perf_counter()
  expanded_nodes = 0
  max_frontier = 1


  while frontiers:
    # tira da fila/fronteira o caminho que vai analisar, por exemplo, 0 - 1 2 3, seleciona o 1 para analisar e tira da fronteira temporariamente
    path = frontiers.popleft()

    # armazena o iltimo estado do path
    state = path[-1]
    expanded_nodes += 1

    if state == GOAL:
      elapsed_time = time.perf_counter() - start_time
      depth = len(path)-1
      return Resposta("bfs", path, depth, expanded_nodes, max_frontier, elapsed_time)


    z = find_zero(state)

    for m in find_valid_moves(z):
      new_state = apply_moves(state, m, z)

      if new_state not in visited:
        visited.add(new_state)
        frontiers.append(path+[new_state])

    max_frontier = max(max_frontier, len(frontiers))

  elapsed_time = round(time.time()-start_time, 2)
  return Resposta("bfs", None, None, expanded_nodes, max_frontier, elapsed_time )

In [ ]:
def dfs(start: tuple) -> Resposta:
# criando atributos de resposta

  start_time = time.perf_counter()
  expanded_nodes = 0
  max_frontier = 1

  # inicializa a pilha
  stack = [[start]]

  while stack:
    max_frontier = max(max_frontier, len(stack))
    path = stack.pop()

    state = path[-1]
    expanded_nodes += 1

    if state == GOAL:
      # encontrou a solução
      elapsed_time = time.perf_counter() - start_time
      depth = len(path)-1
      return Resposta("dfs", path, depth, expanded_nodes, max_frontier, elapsed_time)

    z = find_zero(state)
    for m in find_valid_moves(z):
      new_state = apply_moves(state, m, z)
      # controle de micro estados visitados
      if new_state not in path:
        stack.append(path+[new_state])

  elapsed_time = time.perf_counter() - start_time
  return Resposta("dfs", None, None, expanded_nodes, max_frontier, elapsed_time)



In [ ]:
def dls_limit(state: tuple, path: List[tuple], limit: int, expanded_nodes: int) -> List[int] | None: #
  depth = len(path)-1 # profundidade da arvore
  expanded_nodes += 1

  if state == GOAL:
    return path, expanded_nodes # quebra a recursividade e retorna para a ultima chamda recursiva

  if depth >= limit:
    return None, expanded_nodes # chegou no limite de altura

  z = find_zero(state)
  for m in find_valid_moves(z):
    new_state = apply_moves(state, m, z)

    if new_state not in path:
      result, expanded_nodes = dls_limit(new_state, path+[new_state], limit, expanded_nodes)
      if result is not None: # recursividade voltando
        return result, expanded_nodes # retorna para as recursoes

  return None, expanded_nodes

# Iterative deepening depth-first search
def iddfs(start):
  limit = 0

  # criando atributos de resposta
  start_time = time.perf_counter()
  expanded_nodes = 0
  max_frontier = 1

  while True:
    result, expanded_nodes = dls_limit(start, [start], limit, expanded_nodes)
    max_frontiers = max(max_frontier, expanded_nodes)
    if result is not None:
      elapsed_time = time.perf_counter()-start_time
      depth = len(result)-1
                                                     # max_frontiers?
      return Resposta("iddfs", result, depth, expanded_nodes, max_frontiers, elapsed_time) if result is not None else Resposta("iddfs", None, None, None, None, None)


    limit += 1




In [ ]:
resultados_bfs, resultados_dfs, resultados_iddfs = {}, {}, {}


for caso, estado in casos.items():
    resultados_bfs[caso] = bfs(estado)
    resultados_dfs[caso] = dfs(estado)
    resultados_iddfs[caso] = iddfs(estado)

    respostas = [
        resultados_bfs[caso],
        resultados_dfs[caso],
        resultados_iddfs[caso]
    ]

    for resposta in respostas:
        print(
            f"{caso:<12} | "
            f"{resposta.algoritmo:<10} | "
            f"{str(resposta.depth):<22} | "
            f"{str(resposta.expanded_nodes):<10} | "
            f"{str(resposta.max_frontier):<16} | "
            f"{resposta.time}"
        )
        print()


for resposta in respostas:
      print(
          f"{caso:<12} | "
          f"{resposta.algoritmo:<10} | "
          f"{str(resposta.path)}\n"
      )


Fácil        | bfs        | 1                      | 4          | 5                | 3.0748000426683575e-05

Fácil        | dfs        | 1                      | 2          | 3                | 5.421999958343804e-06

Fácil        | iddfs      | 1                      | 5          | 5                | 7.712998922215775e-06

Médio        | bfs        | 2                      | 7          | 8                | 2.2871001419844106e-05

Médio        | dfs        | 2                      | 3          | 3                | 6.721998943248764e-06

Médio        | iddfs      | 2                      | 11         | 11               | 1.4044000636204146e-05

Desafiador   | bfs        | 4                      | 33         | 28               | 7.894699956523255e-05

Desafiador   | dfs        | 26                     | 27         | 25               | 0.00014326400014397223

Desafiador   | iddfs      | 4                      | 68         | 68               | 8.139199962897692e-05

Desafiador   | bfs      